# పాఠం 11 - మోడల్ కాంటెక్స్ట్ ప్రోటోకాల్ (MCP)

**మోడల్ కాంటెక్స్ట్ ప్రోటోకాల్ (MCP)** అనేది ఓపెన్ స్టాండర్డ్, ఇది ఏజెంట్లు రన్‌టైమ్‌లో డైనమిక్‌గా టూల్స్, వనరులు, మరియు డేటా స్రోతాలను కనుగొని ఉపయోగించడానికి అనుమతిస్తుంది. ఒక ఏజెంట్‌లో టూల్స్‌ను హార్డ్‌కోడ్ చేయడానికి బదులు, MCP ఏజెంట్లు డిమాండ్‌పై సామర్థ్యాలను ప్రదర్శించే బాహ్య సర్వర్లకు కనెక్ట్ అవ్వడానికి మార్గం కల్పిస్తుంది.

ఈ పాఠంలో, మీరు నేర్చుకోనున్నారు:
- MCP ఏమిటి మరియు ఇది ఏజెంట్ సిస్టమ్స్ కోసం ఎందుకు ముఖ్యం
- MCP కస్టమర్-సర్వర్ ఆర్కిటెక్చర్ ఎలా పనిచేస్తుంది
- MCP-శైలి టూల్ కనుగొనటానికి ఉపయోగించే ఏజెంట్లను ఎలా నిర్మించాలో


## సెటప్

**ముందస్తు అవసరాలు:**
- మైక్రోసాఫ్ట్ ఫౌండ్రి ప్రాజెక్ట్‌తో ఒక ప్రతిష్టాపిత మోడల్
- ధృవీకరణ కొరకు `az login` ని అమలు చేయండి


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## మోడల్ కాంటెక్స్ట్ ప్రోటోకాల్ (MCP) అంటే ఏమిటి?

MCP బాహ్య సాధనాలు మరియు డేటా సోర్సులను AI ఏజెంట్లు కనుగొని, వాటితో పరస్పరం చెందడానికి ఒక ప్రమాణ పద్ధతిని నిర్వచిస్తుంది:

- **MCP సర్వర్**: సాధనలు, వనరులు మరియు ప్రాంప్టులను ఒక ప్రమాణ ప్రోటోకాల్ ద్వారా అందిస్తుంది
- **MCP క్లయింట్**: సర్వర్లతో కనెక్ట్ అయ్యి అందుబాటులో ఉన్న సామర్థ్యాలను కనుగొనే ఏజెంట్ రంటైమ్
- **డైనమిక్ డిస్కవరీ**: ఏజెంట్లు హార్డ్‌కోడ్ చేయబడిన సాధనాలను అవసరం లేకుండా, రంటైమ్‌లో అందుబాటులో ఉన్న వాటిని కనుగొంటాయి

ఇది ఏజెంట్ కోడ్‌ని మార్చకుండా కొత్త సామర్థ్యాలను చేర్చగల ఎక్స్‌టెన్సిబుల్ ఏజెంట్ వ్యవస్థలను నిర్మించడానికి శక్తివంతమైనది.


## MCP ఎలా పనిచేస్తుంది

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ఏజెంట్ (MCP క్లయింట్) ఒక MCP సర్వర్ కు కనెక్ట్ అవుతుంది
2. సర్వర్ అందుబాటులో ఉన్న సాధనాలు మరియు వాటి స్కీమాలతో జాబితాను ప్రతిస్పందిస్తుంది
3. ఏజెంట్ తన తర్కాలలో ఏదైనా కనుగొనబడిన సాధనాన్ని పిలవవచ్చు
4. ఫలితాలు అదే ప్రోటోకాల్ ద్వారా తిరిగి వస్తాయి


## MCP టూల్ కనుగొనడం అనుకరించడం

నిజమైన MCP సర్వర్ ఒక రన్నింగ్ సర్వర్ ప్రాసెస్ అవసరం కావడంతో, MCP-కనెక్ట్ అయిన ఆవాస సేవ ఒకటే అందిస్తుందన్ని అనుకరించేందుకు `@tool` ఫంక్షన్లు ఉపయోగించి నమూనాను చూపిస్తాము.

ప్రొడక్షన్‌లో, ఈ టూల్స్ స్థానికంగా నిర్వచించబడటం కాకుండా, MCP సర్వర్ నుండి డైనమిక్గా కనుగొనబడతాయి.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-శైలి టూల్‌లతో ఏజెంట్ నిర్మాణం


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## ఉత్పత్తిలో MCP

ఉత్పత్తి పరిసరంలో, MCP శక్తివంతమైన నమూనాలను సుస్థిరం చేస్తుంది:

- **డైనమిక్ టూల్ కనుగొనడం**: ఏజెంట్లు MCP సర్వర్లతో కనెక్ట్ అవుతూ రన్‌టైంలో టూల్‌లను కనుగొంటారు
- **డీకపిల్ చేసిన ఆర్కిటెక్చర్**: టూల్ ప్రొవైడర్లు ఏజెంట్ నుండి స్వతంత్రంగా అప్డేట్ చేసుకోవచ్చు
- **ఆర్గనైజేషన్ల మాదిరిగా పంచుకోవడం**: జట్లు MCP సర్వర్ల ద్వారా సామర్ధ్యాలను బహిర్గతం చేయగలుగుతాయి, ఏ ఏజెంట్ ఉపయోగించగలదు
- **మైక్రోసాఫ్ట్ ఏజెంట్ ఫ్రేమ్‌వర్క్ మద్దతు**: MAF లో `mcp` ఇంటిగ్రేషన్ ద్వారా బిల్ట్-ఇన్ MCP క్లయింట్ మద్దతు ఉంది

నిజమైన MCP సర్వర్‌ని MAF తో ఉపయోగించడానికి, మీరు `hosted_mcp_tool()` లేదా MCP క్లయింట్ ఇంటిగ్రేషన్ ద్వారా కనెక్ట్ అవ్వాలి.

**ఇంకా తెలుసుకోండి:**
- [MCP స్పెసిఫికేషన్](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP మద్దతు](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## సంక్షేపం

ఈ పాఠంలో, మీరు నేర్చుకున్నది:
- **MCP** ఏజెంట్లు మరియు టూల్ ప్రొవైడర్ల మధ్య డైనమిక్ టూల్ అన్వేషణ కోసం ఓపెన్ స్టాండర్డ్
- **క్లయింట్-సర్వర్ ఆర్కిటెక్చర్** ఏజెంట్లు ఒకరితో సాధనల్ని రన్‌టైమ్‌లో కనుగొనడాన్ని అనుమతిస్తుంది
- MCP **విస్తరించగల, విడివిడి ఏజెంట్ సిస్టమ్స్**ను సాధ్యం చేస్తుంది, ఇక్కడ టూల్స్ కోడ్ మార్పులు లేకుండా జోడించవచ్చు
- మైక్రోసాఫ్ట్ ఏజెంట్ ఫ్రేమ్‌వర్క్ ఉత్పత్తి ఉపయోగానికి **నిక్షిప్త MCP మద్దతును** అందిస్తుంది


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
